In [ ]:
#Performs necessary imports
import pandas as pd
import os
import numpy as np
import itertools
from math import comb
from collections import Counter


In [ ]:

# =====================================================================
# Setup
# =====================================================================
cycles, maxbalance, max_balchange, comb_max, last_election = 5, 10000, 10000, 5, 2020

election_data_raw = pd.read_csv('election_data_2000_2024.csv').fillna(0)
vep_df = pd.read_csv('Turnout_vep.csv', dtype=str)
vep_df = vep_df[vep_df['STATE_ABV'].notna()].copy()
vep_df['VEP'] = vep_df['VEP'].str.replace(',', '', regex=False).astype(float)
vep_df['YEAR'] = vep_df['YEAR'].astype(int)
vep_df = vep_df[['STATE_ABV', 'YEAR', 'VEP']].rename(columns={'STATE_ABV': 'STATE'})

core_states = ['AL', 'AR', 'ID', 'KS', 'KY', 'LA', 'MS', 'ND', 'OK', 'SC', 'SD', 'TN', 'UT',
               'WV', 'WY', 'CA', 'CT', 'DE', 'HI', 'IL', 'MD', 'MA', 'NJ', 'NY', 'OR', 'RI', 'VT', 'WA']


In [ ]:


# =====================================================================
# Building blocks
# =====================================================================
def apply_turnout_shock(election_data_raw, vep_df, core_states, turnout_increase_pct, minority_share_gain):
    scenario_data = election_data_raw.merge(vep_df, on=['STATE', 'YEAR'], how='left')
    is_core = scenario_data['STATE'].isin(core_states)
    d_votes = scenario_data['POPULAR VOTE: D']
    r_votes = scenario_data['POPULAR VOTE: R']
    two_party_total = d_votes + r_votes
    dem_share = (d_votes / two_party_total) * 100
    rep_share = (r_votes / two_party_total) * 100
    dem_is_minority = dem_share < rep_share

    has_vep = scenario_data['VEP'].notna()
    new_voters = np.where(is_core & has_vep, scenario_data['VEP'] * turnout_increase_pct, 0.0)
    new_dem_share = np.where(dem_is_minority, dem_share + minority_share_gain, dem_share - minority_share_gain)
    new_rep_share = np.where(dem_is_minority, rep_share - minority_share_gain, rep_share + minority_share_gain)
    new_dem_share, new_rep_share = np.clip(new_dem_share, 0, 100), np.clip(new_rep_share, 0, 100)

    scenario_data['POPULAR VOTE: D'] += new_voters * (new_dem_share / 100)
    scenario_data['POPULAR VOTE: R'] += new_voters * (new_rep_share / 100)
    return scenario_data.drop(columns=['VEP'])


def build_electiondata(raw_df):
    df = raw_df.copy()
    df['sum_voters'] = df[['POPULAR VOTE: D', 'POPULAR VOTE: R']].sum(axis=1)
    df['DEMOCRAT PERCENTAGE'] = (df['POPULAR VOTE: D'] / df['sum_voters']) * 100
    df['REPUBLICAN PERCENTAGE'] = (df['POPULAR VOTE: R'] / df['sum_voters']) * 100
    df['TOTAL ELECTORAL VOTES FOR EACH STATE'] = df['TOTAL ELECTORAL VOTES: D'] + df['TOTAL ELECTORAL VOTES: R']
    df['q_dem'] = (df['DEMOCRAT PERCENTAGE'] / 100) * df['TOTAL ELECTORAL VOTES FOR EACH STATE']
    df['q_rep'] = (df['REPUBLICAN PERCENTAGE'] / 100) * df['TOTAL ELECTORAL VOTES FOR EACH STATE']

    f_dem, f_rep = np.floor(df['q_dem']).astype(int), np.floor(df['q_rep']).astype(int)
    total_ev = df['TOTAL ELECTORAL VOTES FOR EACH STATE'].round().astype(int)
    remaining = total_ev - f_dem - f_rep
    dem_gets_leftover = (remaining > 0) & ((df['q_dem'] - f_dem) >= (df['q_rep'] - f_rep))

    df['DEMOCRAT REALLOCATED'] = f_dem + np.where(dem_gets_leftover, remaining, 0)
    df['REPUBLICAN REALLOCATED'] = f_rep + np.where((remaining > 0) & ~dem_gets_leftover, remaining, 0)
    return df


def build_orig_pres_results(electiondata):
    dem_total = electiondata.groupby('YEAR')['TOTAL ELECTORAL VOTES: D'].sum().reset_index()
    dem_total.columns = ['YEAR', 'TOTAL_DEM_VOTES']
    rep_total = electiondata.groupby('YEAR')['TOTAL ELECTORAL VOTES: R'].sum().reset_index()
    rep_total.columns = ['YEAR', 'TOTAL_REP_VOTES']
    merged_df = dem_total.merge(rep_total, on='YEAR')
    merged_df['DIFF'] = merged_df['TOTAL_DEM_VOTES'] - merged_df['TOTAL_REP_VOTES']
    merged_df['WINNER'] = np.where(merged_df['DIFF'] > 0, 'D', 'R')
    merged_df['VOTES_TO_WIN'] = np.floor((merged_df['DIFF'].abs() / 2) + 1)
    return merged_df


# =====================================================================
#  always-stable counter -- processes combinations in chunks,
# checking every year for each chunk before moving on, and NEVER stores
# a DataFrame or list of all combinations. Peak memory is bounded by
# chunk_size x combination_size, regardless of how many total
# combinations exist (millions for 5-state).
# =====================================================================
def count_always_stable_lowmem(electiondata, comb_max, cycles, orig_pres_results,
                                maxbalance, max_balchange, last_election=2020, chunk_size=200_000):
    min_year = last_election - 4 * (cycles - 1)
    edf = electiondata[electiondata['YEAR'] >= min_year]
    years = sorted(edf['YEAR'].unique())

    year_data = {}
    for year, ydf in edf.groupby('YEAR'):
        ydf = ydf.sort_values('STATE')
        winner_row = orig_pres_results.loc[orig_pres_results['YEAR'] == year].iloc[0]
        year_data[year] = {
            'dem': ydf['TOTAL ELECTORAL VOTES: D'].to_numpy(),
            'rep': ydf['TOTAL ELECTORAL VOTES: R'].to_numpy(),
            'pdem': ydf['DEMOCRAT REALLOCATED'].to_numpy(),
            'prep': ydf['REPUBLICAN REALLOCATED'].to_numpy(),
            'winner': winner_row['WINNER'],
            'votes_to_win': winner_row['VOTES_TO_WIN'],
        }

    n = len(year_data[years[0]]['dem'])
    counts_by_size = {}

    for r in range(3, comb_max + 1):
        if r > n:
            continue
        total_qualify = 0
        comb_iter = itertools.combinations(range(n), r)
        while True:
            chunk = list(itertools.islice(comb_iter, chunk_size))
            if not chunk:
                break
            idx = np.array(chunk, dtype=np.int64)
            qualifies = np.ones(len(idx), dtype=bool)

            for year in years:
                if not qualifies.any():
                    break  # nothing left in this chunk can qualify -- skip remaining years
                yd = year_data[year]
                d, rp = yd['dem'][idx].sum(axis=1), yd['rep'][idx].sum(axis=1)
                pd_, pr = yd['pdem'][idx].sum(axis=1), yd['prep'][idx].sum(axis=1)

                balance = np.abs(d - rp)
                balance_prop = np.abs(pd_ - pr)
                flip = (d > rp) != (pd_ > pr)
                balance_change = np.where(flip, balance + balance_prop, np.abs(balance - balance_prop))

                passes_filter = (balance <= maxbalance) & (balance_change <= max_balchange)
                if yd['winner'] == 'D':
                    stable = ~(pr - rp > yd['votes_to_win'])   # chg_rep = pr - rp
                else:
                    stable = ~(pd_ - d > yd['votes_to_win'])   # chg_dem = pd_ - d

                qualifies &= passes_filter & stable

            total_qualify += int(qualifies.sum())
            del idx, qualifies  # chunk is discarded here regardless

        counts_by_size[r] = total_qualify

    return counts_by_size

def counts_to_number_and_pct(counts_by_size, n_states):
    out = {}
    for size, count in counts_by_size.items():
        total_combos = comb(n_states, size)
        pct = count / total_combos * 100 if total_combos else 0.0
        out[f'{size}state_always_stable'] = count
        out[f'{size}state_total_combinations'] = total_combos
        out[f'{size}state_pct_always_stable'] = round(pct, 3)
    return out


electiondata_baseline = build_electiondata(election_data_raw)
orig_pres_results = build_orig_pres_results(electiondata_baseline)
n_states = electiondata_baseline['STATE'].nunique()

baseline_counts = count_always_stable_lowmem(electiondata_baseline, comb_max, cycles,
                                              orig_pres_results, maxbalance, max_balchange, last_election)
baseline_report = counts_to_number_and_pct(baseline_counts, n_states)

print(f"Number of distinct states/jurisdictions: {n_states}\n")
print("=== ORIGINAL (unshocked) data: number and percentage always proportional-stable ===")
for size in sorted(baseline_counts):
    print(f"  {size}-state: {baseline_report[f'{size}state_always_stable']:,} of "
          f"{baseline_report[f'{size}state_total_combinations']:,} "
          f"({baseline_report[f'{size}state_pct_always_stable']:.2f}%)")

Number of distinct states/jurisdictions: 51

=== ORIGINAL (unshocked) data: number and percentage always proportional-stable ===
  3-state: 20,258 of 20,825 (97.28%)
  4-state: 237,029 of 249,900 (94.85%)
  5-state: 2,167,897 of 2,349,060 (92.29%)


In [ ]:
# =====================================================================
# Sensitivity grid across shock sizes, for sizes 3, 4, and 5 together --
# now reports number AND percentage for both baseline and each scenario
# =====================================================================
records = []

# Row 0: the baseline itself, so it's visible alongside every scenario in the same table
baseline_row = {'turnout_increase_pct': 0.0, 'minority_share_gain': 0.0}
baseline_row.update(baseline_report)
records.append(baseline_row)

for turnout_pct in [0.01, 0.05, 0.10, 0.20]:
    for lean in [2.0, 5.0, 10.0]:
        scenario_data = apply_turnout_shock(election_data_raw, vep_df, core_states, turnout_pct, lean)
        electiondata_scenario = build_electiondata(scenario_data)
        scenario_counts = count_always_stable_lowmem(electiondata_scenario, comb_max, cycles,
                                                       orig_pres_results, maxbalance, max_balchange, last_election)
        scenario_report = counts_to_number_and_pct(scenario_counts, n_states)

        row = {'turnout_increase_pct': turnout_pct, 'minority_share_gain': lean}
        row.update(scenario_report)
        for size in sorted(baseline_counts):
            row[f'{size}state_net_change'] = scenario_counts.get(size, 0) - baseline_counts[size]
            row[f'{size}state_pct_point_change'] = round(
                scenario_report[f'{size}state_pct_always_stable'] - baseline_report[f'{size}state_pct_always_stable'], 4
            )
        records.append(row)
        del scenario_data, electiondata_scenario, scenario_counts, scenario_report  # free before next iteration

sensitivity_df = pd.DataFrame(records)
print("\n=== Full table: original + shocked data, number and percentage always proportional-stable ===")
print(sensitivity_df.to_string(index=False))
sensitivity_df.to_csv('always_stable_sensitivity_3_4_5.csv', index=False)


=== Full table: original + shocked data, number and percentage always proportional-stable ===
 turnout_increase_pct  minority_share_gain  3state_always_stable  3state_total_combinations  3state_pct_always_stable  4state_always_stable  4state_total_combinations  4state_pct_always_stable  5state_always_stable  5state_total_combinations  5state_pct_always_stable  3state_net_change  3state_pct_point_change  4state_net_change  4state_pct_point_change  5state_net_change  5state_pct_point_change
                 0.00                  0.0                 20258                      20825                    97.277                237029                     249900                    94.850               2167897                    2349060                    92.288                NaN                      NaN                NaN                      NaN                NaN                      NaN
                 0.01                  2.0                 20258                      20825              

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_excel('always_stable_sensitivity_2004_2020.csv')

# Separate baseline (turnout=0) from the actual shock scenarios
baseline = df[df['turnout_increase_pct'] == 0].iloc[0]
scenarios = df[df['turnout_increase_pct'] > 0].copy()

sizes = [3, 4, 5]
turnout_vals = sorted(scenarios['turnout_increase_pct'].unique())
lean_vals = sorted(scenarios['minority_share_gain'].unique())

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

# Shared color scale across all 3 panels, symmetric around 0 so the
# diverging colormap is centered correctly everywhere
vmax = scenarios[[f'{s}state_pct_point_change' for s in sizes]].abs().max().max()

for ax, s in zip(axes, sizes):
    col = f'{s}state_pct_point_change'
    pivot = scenarios.pivot(index='turnout_increase_pct', columns='minority_share_gain', values=col)
    pivot = pivot.loc[turnout_vals, lean_vals]

    im = ax.imshow(pivot.values, cmap='RdBu_r', vmin=-vmax, vmax=vmax, aspect='auto')

    ax.set_xticks(range(len(lean_vals)))
    ax.set_xticklabels([f'{v:.0f}pt' for v in lean_vals])
    ax.set_yticks(range(len(turnout_vals)))
    ax.set_yticklabels([f'{v*100:.0f}%' for v in turnout_vals])
    ax.set_xlabel('minority-party lean among new voters')
    if s == 3:
        ax.set_ylabel('turnout increase (% of VEP)')
    ax.set_title(f'{s}-state combinations')

    # Annotate each cell with its value, flipping text color for readability
    # on the darkest cells
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            ax.text(j, i, f'{val:+.3f}', ha='center', va='center', fontsize=9,
                     color='white' if abs(val) > vmax * 0.6 else 'black')

fig.colorbar(im, ax=axes, shrink=0.7, label='Change in % always-stable\n(percentage points vs. baseline)')
fig.suptitle('Sensitivity of proportional-stability rate to turnout shock scenarios (2004\u20132020)', y=1.05)
plt.savefig('sensitivity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()